### This notebook evaluates Transformer variants beyond the baseline.

#### Evaluation metric: sMAPE

#### Forecast horizons tested: 24-step and 96-step

#### Residual modeling retained (because baseline showed improvement)

##### Naive baseline computed for each horizon

In [19]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import math
import pandas as pd



In [20]:
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print("Using device:", device)


Using device: mps


In [21]:
df_train = pd.read_csv("../data/raw/train_data.csv")
df_val = pd.read_csv("../data/raw/validation_data.csv")

FEATURE_COLS = [c for c in df_train.columns if c != "timestamp_idx"]

train_data = df_train[FEATURE_COLS].values
val_data = df_val[FEATURE_COLS].values

print("Train shape:", train_data.shape)


Train shape: (5311, 9)


In [22]:
def create_windows(data, seq_len, pred_len):
    X, Y = [], []
    for i in range(len(data) - seq_len - pred_len + 1):
        X.append(data[i:i+seq_len])
        Y.append(data[i+seq_len:i+seq_len+pred_len])
    return np.array(X), np.array(Y)


In [23]:
class TimeSeriesDataset(Dataset):
    def __init__(self, X, Y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.Y = torch.tensor(Y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.Y[idx]


In [24]:
def smape(y_true, y_pred):
    eps = 1e-8
    return torch.mean(
        2 * torch.abs(y_true - y_pred) /
        (torch.abs(y_true) + torch.abs(y_pred) + eps)
    )


In [25]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()

        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model)
        )

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        pe = pe.unsqueeze(0)
        self.register_buffer("pe", pe)

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]


In [26]:
def compute_naive(X, Y):
    X = torch.tensor(X, dtype=torch.float32).to(device)
    Y = torch.tensor(Y, dtype=torch.float32).to(device)

    last_value = X[:, -1:, :]
    naive_pred = last_value.repeat(1, Y.shape[1], 1)

    return smape(Y, naive_pred).item()


In [27]:
SEQ_LEN = 96
PRED_LEN = 24

X_train, Y_train = create_windows(train_data, SEQ_LEN, PRED_LEN)
X_val, Y_val     = create_windows(val_data, SEQ_LEN, PRED_LEN)

Y_train_res = Y_train - X_train[:, -1:, :]
Y_val_res   = Y_val - X_val[:, -1:, :]


In [28]:
naive_24 = compute_naive(X_val, Y_val)
print("Naive 24-step sMAPE:", naive_24)


Naive 24-step sMAPE: 0.07527655363082886


#### Informer:


In [29]:
SEQ_LEN = 96
PRED_LEN = 24
input_dim = train_data.shape[1]

D_MODEL = 64
N_HEADS = 4
ENC_LAYERS = 2
DROPOUT = 0.1


In [30]:
X_train, Y_train = create_windows(train_data, SEQ_LEN, PRED_LEN)
X_val, Y_val     = create_windows(val_data, SEQ_LEN, PRED_LEN)

# Residual modeling
Y_train_res = Y_train - X_train[:, -1:, :]
Y_val_res   = Y_val - X_val[:, -1:, :]


In [31]:
train_dataset = TimeSeriesDataset(X_train, Y_train_res)
val_dataset   = TimeSeriesDataset(X_val, Y_val_res)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=False)
val_loader   = DataLoader(val_dataset, batch_size=16, shuffle=False)


In [32]:
class InformerEncoderOnly(nn.Module):
    def __init__(self):
        super().__init__()

        self.input_proj = nn.Linear(input_dim, D_MODEL)
        self.pos_enc = PositionalEncoding(D_MODEL)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=D_MODEL,
            nhead=N_HEADS,
            dropout=DROPOUT,
            batch_first=True
        )

        self.encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=ENC_LAYERS
        )

        # 🔹 Distillation layer (reduces sequence length)
        self.distill = nn.Conv1d(
            in_channels=D_MODEL,
            out_channels=D_MODEL,
            kernel_size=3,
            padding=1,
            stride=2
        )

        self.output_proj = nn.Linear(D_MODEL, PRED_LEN * input_dim)

    def forward(self, x):

        x = self.input_proj(x)
        x = self.pos_enc(x)
        x = self.encoder(x)

        # Distillation
        x = x.transpose(1, 2)      # (B, D, T)
        x = self.distill(x)
        x = x.transpose(1, 2)      # (B, T/2, D)

        x = x[:, -1, :]            # Last compressed timestep

        out = self.output_proj(x)
        out = out.view(-1, PRED_LEN, input_dim)

        return out


In [33]:
def train_model(model, train_loader, val_loader, residual=False):

    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
    criterion = nn.MSELoss()

    best_val = float("inf")
    patience=5
    patience_counter=0


    for epoch in range(20):
        model.train()
        train_loss = 0

        for X_batch, Y_batch in train_loader:
            X_batch = X_batch.to(device)
            Y_batch = Y_batch.to(device)

            optimizer.zero_grad()
            output = model(X_batch)

            loss = criterion(output, Y_batch)
            loss.backward()

            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            train_loss += loss.item()

        # Validation
        model.eval()
        val_smape = 0

        with torch.no_grad():
            for X_batch, Y_batch_res in val_loader:

                X_batch = X_batch.to(device)
                Y_batch_res = Y_batch_res.to(device)

                residual_pred = model(X_batch)

                last_value = X_batch[:, -1:, :]
                final_pred = last_value + residual_pred
                Y_original = last_value + Y_batch_res

                val_smape += smape(Y_original, final_pred).item()

        val_smape /= len(val_loader)

        print(f"Epoch {epoch+1} | Train Loss: {train_loss/len(train_loader):.4f} | Val sMAPE: {val_smape:.4f}")

        if val_smape<best_val:
            best_val=val_smape
            patience_counter=0
            torch.save(model.state_dict(),"best_model.pth")
        else:
            patience_counter+=1

        if patience_counter>=patience:
            print("Early stopping triggered")
            break
    model.load_state_dict(torch.load("best_model.pth"))
    return best_val








    


In [34]:
model = InformerEncoderOnly().to(device)

val_smape_informer = train_model(
    model,
    train_loader,
    val_loader,
    residual=True
)

print("Informer (24-step) Val sMAPE:", val_smape_informer)


Epoch 1 | Train Loss: 9.6246 | Val sMAPE: 0.0872
Epoch 2 | Train Loss: 0.0390 | Val sMAPE: 0.0785
Epoch 3 | Train Loss: 0.0323 | Val sMAPE: 0.0766
Epoch 4 | Train Loss: 0.0316 | Val sMAPE: 0.0762
Epoch 5 | Train Loss: 0.0313 | Val sMAPE: 0.0749
Epoch 6 | Train Loss: 0.0311 | Val sMAPE: 0.0749
Epoch 7 | Train Loss: 0.0309 | Val sMAPE: 0.0748
Epoch 8 | Train Loss: 0.0308 | Val sMAPE: 0.0744
Epoch 9 | Train Loss: 0.0306 | Val sMAPE: 0.0744
Epoch 10 | Train Loss: 0.0305 | Val sMAPE: 0.0741
Epoch 11 | Train Loss: 0.0304 | Val sMAPE: 0.0739
Epoch 12 | Train Loss: 0.0303 | Val sMAPE: 0.0740
Epoch 13 | Train Loss: 0.0302 | Val sMAPE: 0.0739
Epoch 14 | Train Loss: 0.0301 | Val sMAPE: 0.0735
Epoch 15 | Train Loss: 0.0300 | Val sMAPE: 0.0733
Epoch 16 | Train Loss: 0.0299 | Val sMAPE: 0.0731
Epoch 17 | Train Loss: 0.0298 | Val sMAPE: 0.0730
Epoch 18 | Train Loss: 0.0298 | Val sMAPE: 0.0728
Epoch 19 | Train Loss: 0.0297 | Val sMAPE: 0.0726
Epoch 20 | Train Loss: 0.0296 | Val sMAPE: 0.0725
Informer 

#### Autoformer:


In [35]:
SEQ_LEN = 96
PRED_LEN = 24
input_dim = train_data.shape[1]

D_MODEL = 64
N_HEADS = 4
ENC_LAYERS = 2
DROPOUT = 0.1


In [36]:
class SeriesDecomposition(nn.Module):
    def __init__(self, kernel_size=25):
        super().__init__()
        self.kernel_size = kernel_size
        self.padding = kernel_size // 2

    def forward(self, x):
        # Moving average (trend)
        trend = nn.functional.avg_pool1d(
            x.transpose(1, 2),
            kernel_size=self.kernel_size,
            stride=1,
            padding=self.padding
        ).transpose(1, 2)

        seasonal = x - trend
        return seasonal, trend


In [37]:
class AutoformerEncoderOnly(nn.Module):
    def __init__(self,
                 input_dim,
                 pred_len,
                 kernel_size,
                 d_model,
                 n_heads,
                 enc_layers,
                 dropout):

        super().__init__()

        self.pred_len = pred_len
        self.input_dim = input_dim

        self.decomp = SeriesDecomposition(kernel_size)

        self.input_proj = nn.Linear(input_dim, d_model)
        self.pos_enc = PositionalEncoding(d_model)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dropout=dropout,
            batch_first=True
        )

        self.encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=enc_layers
        )

        self.output_proj = nn.Linear(d_model, pred_len * input_dim)

    def forward(self, x):

        seasonal, trend = self.decomp(x)

        x = self.input_proj(seasonal)
        x = self.pos_enc(x)
        x = self.encoder(x)

        x = x[:, -1, :]

        out = self.output_proj(x)
        out = out.view(-1, self.pred_len, self.input_dim)

        return out


In [38]:
X_train, Y_train = create_windows(train_data, SEQ_LEN, PRED_LEN)
X_val, Y_val     = create_windows(val_data, SEQ_LEN, PRED_LEN)

# Residual modeling
Y_train_res = Y_train - X_train[:, -1:, :]
Y_val_res   = Y_val - X_val[:, -1:, :]


In [39]:
train_dataset = TimeSeriesDataset(X_train, Y_train_res)
val_dataset   = TimeSeriesDataset(X_val, Y_val_res)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=False)
val_loader   = DataLoader(val_dataset, batch_size=16, shuffle=False)


In [73]:
def train_model(model, train_loader, val_loader, residual=False):

    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
    criterion = nn.MSELoss()

    best_val = float("inf")
    patience=5
    patience_counter=0


    for epoch in range(20):
        model.train()
        train_loss = 0

        for X_batch, Y_batch in train_loader:
            X_batch = X_batch.to(device)
            Y_batch = Y_batch.to(device)

            optimizer.zero_grad()
            output = model(X_batch)

            loss = criterion(output, Y_batch)
            loss.backward()

            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            train_loss += loss.item()

        # Validation
        model.eval()
        val_smape = 0

        with torch.no_grad():
            for X_batch, Y_batch_res in val_loader:

                X_batch = X_batch.to(device)
                Y_batch_res = Y_batch_res.to(device)

                residual_pred = model(X_batch)

                last_value = X_batch[:, -1:, :]
                final_pred = last_value + residual_pred
                Y_original = last_value + Y_batch_res

                val_smape += smape(Y_original, final_pred).item()

        val_smape /= len(val_loader)

        print(f"Epoch {epoch+1} | Train Loss: {train_loss/len(train_loader):.4f} | Val sMAPE: {val_smape:.4f}")

        if val_smape<best_val:
            best_val=val_smape
            patience_counter=0
            torch.save(model.state_dict(),"best_model.pth")
        else:
            patience_counter+=1

        if patience_counter>=patience:
            print("Early stopping triggered")
            break
    model.load_state_dict(torch.load("best_model.pth"))
    return best_val








  


In [41]:
model = AutoformerEncoderOnly().to(device)

val_smape_autoformer = train_model(
    model,
    train_loader,
    val_loader,
    residual=True
)

print("Autoformer (24-step) Val sMAPE:", val_smape_autoformer)


Epoch 1 | Train Loss: 17.3726 | Val sMAPE: 0.0764
Epoch 2 | Train Loss: 11.7110 | Val sMAPE: 0.0739
Epoch 3 | Train Loss: 6.9817 | Val sMAPE: 0.0733
Epoch 4 | Train Loss: 3.4573 | Val sMAPE: 0.0728
Epoch 5 | Train Loss: 1.2029 | Val sMAPE: 0.0729
Epoch 6 | Train Loss: 0.2013 | Val sMAPE: 0.0768
Epoch 7 | Train Loss: 0.0416 | Val sMAPE: 0.0736
Epoch 8 | Train Loss: 0.0299 | Val sMAPE: 0.0738
Epoch 9 | Train Loss: 0.0291 | Val sMAPE: 0.0745
Early stopping triggered
Autoformer (24-step) Val sMAPE: 0.07282309252768755


In [42]:
results={}
results["Informer 24"] = {
    "model_smape": val_smape_informer,
    "naive_smape": naive_24
}

results["Autoformer 24"] = {
    "model_smape": val_smape_autoformer,
    "naive_smape": naive_24
}


In [43]:
import pandas as pd
comparison=pd.DataFrame(results).T
comparison["beats_naive"]=comparison["model_smape"]<comparison["naive_smape"]
comparison.round(4)

,model_smape,naive_smape,beats_naive
Informer 24,0.0725,0.0753,True
Autoformer 24,0.0728,0.0753,True


### Longer Horizon : Prediction length-96

In [44]:
SEQ_LEN = 96
PRED_LEN = 96 #prdiction window updated
input_dim = train_data.shape[1]

D_MODEL = 64
N_HEADS = 4
ENC_LAYERS = 2
DROPOUT = 0.1


In [45]:
X_train, Y_train = create_windows(train_data, SEQ_LEN, PRED_LEN)
X_val, Y_val     = create_windows(val_data, SEQ_LEN, PRED_LEN)

Y_train_res = Y_train - X_train[:, -1:, :]
Y_val_res   = Y_val - X_val[:, -1:, :]


In [46]:
train_dataset = TimeSeriesDataset(X_train, Y_train_res)
val_dataset   = TimeSeriesDataset(X_val, Y_val_res)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=False)
val_loader   = DataLoader(val_dataset, batch_size=16, shuffle=False)


### Naive baseline for new pred_window

In [47]:
naive_96 = compute_naive(X_val, Y_val)
print("Naive 96-step sMAPE:", naive_96)


Naive 96-step sMAPE: 0.13783612847328186


In [49]:
model = InformerEncoderOnly().to(device)
val_smape_informer_96 = train_model(model, train_loader, val_loader, residual=True)

model = AutoformerEncoderOnly().to(device)
val_smape_autoformer_96 = train_model(model, train_loader, val_loader, residual=True)


Epoch 1 | Train Loss: 283.1799 | Val sMAPE: 0.1351
Epoch 2 | Train Loss: 79.9931 | Val sMAPE: 0.1600
Epoch 3 | Train Loss: 0.8718 | Val sMAPE: 0.1391
Epoch 4 | Train Loss: 0.1480 | Val sMAPE: 0.1281
Epoch 5 | Train Loss: 0.1411 | Val sMAPE: 0.1263
Epoch 6 | Train Loss: 0.1389 | Val sMAPE: 0.1254
Epoch 7 | Train Loss: 0.1377 | Val sMAPE: 0.1251
Epoch 8 | Train Loss: 0.1370 | Val sMAPE: 0.1250
Epoch 9 | Train Loss: 0.1365 | Val sMAPE: 0.1250
Epoch 10 | Train Loss: 0.1361 | Val sMAPE: 0.1247
Epoch 11 | Train Loss: 0.1358 | Val sMAPE: 0.1246
Epoch 12 | Train Loss: 0.1354 | Val sMAPE: 0.1246
Epoch 13 | Train Loss: 0.1351 | Val sMAPE: 0.1244
Epoch 14 | Train Loss: 0.1348 | Val sMAPE: 0.1244
Epoch 15 | Train Loss: 0.1344 | Val sMAPE: 0.1243
Epoch 16 | Train Loss: 0.1342 | Val sMAPE: 0.1241
Epoch 17 | Train Loss: 0.1337 | Val sMAPE: 0.1240
Epoch 18 | Train Loss: 0.1335 | Val sMAPE: 0.1239
Epoch 19 | Train Loss: 0.1332 | Val sMAPE: 0.1237
Epoch 20 | Train Loss: 0.1328 | Val sMAPE: 0.1236
Epoch 

In [50]:
results["Informer 96"] = {
    "model_smape": val_smape_informer_96,
    "naive_smape": naive_96
}

results["Autoformer 96"] = {
    "model_smape": val_smape_autoformer_96,
    "naive_smape": naive_96
}


In [51]:
comparison=pd.DataFrame(results).T
comparison["beats_naive"]=comparison["model_smape"]<comparison["naive_smape"]
comparison.round(4)


,model_smape,naive_smape,beats_naive
Informer 24,0.0725,0.0753,True
Autoformer 24,0.0728,0.0753,True
Informer 96,0.1236,0.1378,True
Autoformer 96,0.1224,0.1378,True


#### Predition Horizon :199

In [52]:
PRED_LEN = 199


In [53]:
X_train, Y_train = create_windows(train_data, SEQ_LEN, PRED_LEN)
X_val, Y_val     = create_windows(val_data, SEQ_LEN, PRED_LEN)

Y_train_res = Y_train - X_train[:, -1:, :]
Y_val_res   = Y_val - X_val[:, -1:, :]


In [54]:
train_dataset = TimeSeriesDataset(X_train, Y_train_res)
val_dataset   = TimeSeriesDataset(X_val, Y_val_res)


In [ ]:
train_dataset = TimeSeriesDataset(X_train, Y_train_res)
val_dataset   = TimeSeriesDataset(X_val, Y_val_res)



train_loader = DataLoader(train_dataset, batch_size=16, shuffle=False)
val_loader   = DataLoader(val_dataset, batch_size=16, shuffle=False)


In [56]:
print("Train samples:", X_train.shape[0])
print("Val samples:", X_val.shape[0])


Train samples: 5017
Val samples: 465


In [57]:
naive_199=compute_naive(X_val,Y_val)
print("Naive 199-step sMAPE:", naive_199)

Naive 199-step sMAPE: 0.1933424025774002


In [59]:
model=InformerEncoderOnly().to(device)
val_smape_informer_199=train_model(model,train_loader,val_loader,residual=True)


model=AutoformerEncoderOnly().to(device)
val_smape_autoformer_199=train_model(model, train_loader,val_loader,residual=True)

Epoch 1 | Train Loss: 1344.1794 | Val sMAPE: 0.2059
Epoch 2 | Train Loss: 786.2973 | Val sMAPE: 0.2569
Epoch 3 | Train Loss: 196.3206 | Val sMAPE: 0.3435
Epoch 4 | Train Loss: 2.8965 | Val sMAPE: 0.2737
Epoch 5 | Train Loss: 0.3431 | Val sMAPE: 0.1939
Epoch 6 | Train Loss: 0.3201 | Val sMAPE: 0.1851
Epoch 7 | Train Loss: 0.3127 | Val sMAPE: 0.1826
Epoch 8 | Train Loss: 0.3095 | Val sMAPE: 0.1811
Epoch 9 | Train Loss: 0.3073 | Val sMAPE: 0.1803
Epoch 10 | Train Loss: 0.3061 | Val sMAPE: 0.1796
Epoch 11 | Train Loss: 0.3051 | Val sMAPE: 0.1791
Epoch 12 | Train Loss: 0.3041 | Val sMAPE: 0.1786
Epoch 13 | Train Loss: 0.3035 | Val sMAPE: 0.1783
Epoch 14 | Train Loss: 0.3027 | Val sMAPE: 0.1779
Epoch 15 | Train Loss: 0.3019 | Val sMAPE: 0.1776
Epoch 16 | Train Loss: 0.3010 | Val sMAPE: 0.1774
Epoch 17 | Train Loss: 0.3003 | Val sMAPE: 0.1770
Epoch 18 | Train Loss: 0.2996 | Val sMAPE: 0.1767
Epoch 19 | Train Loss: 0.2987 | Val sMAPE: 0.1765
Epoch 20 | Train Loss: 0.2980 | Val sMAPE: 0.1763
Ep

In [60]:
results["Informer 199"] = {
    "model_smape": val_smape_informer_199,
    "naive_smape": naive_199
}

results["Autoformer 199"] = {
    "model_smape": val_smape_autoformer_199,
    "naive_smape": naive_199
}


In [61]:
comparison =pd.DataFrame(results).T
comparison["beats_naive"]=comparison["model_smape"]<comparison["naive_smape"]
comparison.round(4)

,model_smape,naive_smape,beats_naive
Informer 24,0.0725,0.0753,True
Autoformer 24,0.0728,0.0753,True
Informer 96,0.1236,0.1378,True
Autoformer 96,0.1224,0.1378,True
Informer 199,0.1763,0.1933,True
Autoformer 199,0.1608,0.1933,True


#### Prediction Horizon :366

In [62]:
PRED_LEN=366

In [63]:
X_train, Y_train = create_windows(train_data, SEQ_LEN, PRED_LEN)
X_val, Y_val     = create_windows(val_data, SEQ_LEN, PRED_LEN)

Y_train_res = Y_train - X_train[:, -1:, :]
Y_val_res   = Y_val - X_val[:, -1:, :]


In [65]:
train_dataset = TimeSeriesDataset(X_train, Y_train_res)
val_dataset   = TimeSeriesDataset(X_val, Y_val_res)


In [66]:
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=False)
val_loader   = DataLoader(val_dataset, batch_size=16, shuffle=False)


In [67]:
print("Train samples:", X_train.shape[0])
print("Val samples:", X_val.shape[0])


Train samples: 4850
Val samples: 298


In [68]:
naive_366=compute_naive(X_val,Y_val)
print("Naive 366-step sMAPE:", naive_366)

Naive 366-step sMAPE: 0.25456294417381287


In [70]:
model=InformerEncoderOnly().to(device)
val_smape_informer_366=train_model(model,train_loader,val_loader,residual=True)


model=AutoformerEncoderOnly().to(device)
val_smape_autoformer_366=train_model(model, train_loader,val_loader,residual=True)

Epoch 1 | Train Loss: 4773.0486 | Val sMAPE: 0.2898
Epoch 2 | Train Loss: 3740.2830 | Val sMAPE: 0.3507
Epoch 3 | Train Loss: 2219.6336 | Val sMAPE: 0.2857
Epoch 4 | Train Loss: 805.0387 | Val sMAPE: 0.5208
Epoch 5 | Train Loss: 71.4526 | Val sMAPE: 0.4794
Epoch 6 | Train Loss: 0.5287 | Val sMAPE: 0.3824
Epoch 7 | Train Loss: 0.4669 | Val sMAPE: 0.3881
Epoch 8 | Train Loss: 0.4599 | Val sMAPE: 0.3942
Early stopping triggered
Epoch 1 | Train Loss: 4931.5619 | Val sMAPE: 0.2372
Epoch 2 | Train Loss: 4842.8923 | Val sMAPE: 0.2284
Epoch 3 | Train Loss: 4755.3869 | Val sMAPE: 0.2267
Epoch 4 | Train Loss: 4666.4117 | Val sMAPE: 0.2250
Epoch 5 | Train Loss: 4574.0460 | Val sMAPE: 0.2232
Epoch 6 | Train Loss: 4477.2635 | Val sMAPE: 0.2212
Epoch 7 | Train Loss: 4375.4339 | Val sMAPE: 0.2193
Epoch 8 | Train Loss: 4268.2817 | Val sMAPE: 0.2173
Epoch 9 | Train Loss: 4155.6471 | Val sMAPE: 0.2154
Epoch 10 | Train Loss: 4037.5249 | Val sMAPE: 0.2135
Epoch 11 | Train Loss: 3913.9887 | Val sMAPE: 0.21

In [71]:
results["Informer 366"] = {
    "model_smape": val_smape_informer_366,
    "naive_smape": naive_366
}

results["Autoformer 366"] = {
    "model_smape": val_smape_autoformer_366,
    "naive_smape": naive_366
}


In [72]:
comparison =pd.DataFrame(results).T
comparison["beats_naive"]=comparison["model_smape"]<comparison["naive_smape"]
comparison.round(4)

,model_smape,naive_smape,beats_naive
Informer 24,0.0725,0.0753,True
Autoformer 24,0.0728,0.0753,True
Informer 96,0.1236,0.1378,True
Autoformer 96,0.1224,0.1378,True
Informer 199,0.1763,0.1933,True
Autoformer 199,0.1608,0.1933,True
Informer 366,0.2857,0.2546,False
Autoformer 366,0.1988,0.2546,True


## The Informer and Autoformer experiments for different horizons(keeping the rest constant ) are all done with Autoformer performing the best .

## Now let's move to experiments where the input window length changes (SEQ_LEN) and experiment with the kernel size and attendtion heads as well 

In [94]:
def config_setup(seq_len,pred_len,input_dims,d_model,n_heads,enc_layers,dropout):
    SEQ_LEN = seq_len
    PRED_LEN = pred_len #prdiction window updated
    input_dim = input_dims

    D_MODEL = d_model
    N_HEADS = n_heads
    ENC_LAYERS = enc_layers
    DROPOUT = dropout
    return(SEQ_LEN,PRED_LEN,input_dim,D_MODEL,N_HEADS,ENC_LAYERS,DROPOUT)


In [100]:
def windowcreation_caller(seq_len, pred_len):
    X_train, Y_train = create_windows(train_data, seq_len, pred_len)
    X_val, Y_val     = create_windows(val_data, seq_len, pred_len)

    Y_train_res = Y_train - X_train[:, -1:, :]
    Y_val_res   = Y_val - X_val[:, -1:, :]
    return (X_train,Y_train_res,X_val, Y_val_res,Y_val)

In [93]:
def dataloader_call(x_train,y_train_res,x_val,y_val_res):
    train_dataset = TimeSeriesDataset(x_train, y_train_res)
    val_dataset   = TimeSeriesDataset(x_val, y_val_res)

    train_loader = DataLoader(train_dataset, batch_size=16, shuffle=False)
    val_loader   = DataLoader(val_dataset, batch_size=16, shuffle=False)
    return (train_loader,val_loader)


In [141]:
#Model rebuild and initialise :

class InformerEncoderOnly(nn.Module):
    def __init__(self, input_dim,pred_len,d_model,n_heads,enc_layers,dropout):
        super().__init__()

        self.pred_len=pred_len
        self.input_dim=input_dim


        self.input_proj = nn.Linear(input_dim, d_model)
        self.pos_enc = PositionalEncoding(d_model)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dropout=dropout,
            batch_first=True
        )

        self.encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=enc_layers
        )

        # 🔹 Distillation layer (reduces sequence length)
        self.distill = nn.Conv1d(
            in_channels=d_model,
            out_channels=d_model,
            kernel_size=3,
            padding=1,
            stride=2
        )

        self.output_proj = nn.Linear(d_model, pred_len * input_dim)

    def forward(self, x):

        x = self.input_proj(x)
        x = self.pos_enc(x)
        x = self.encoder(x)

        # Distillation
        x = x.transpose(1, 2)      # (B, D, T)
        x = self.distill(x)
        x = x.transpose(1, 2)      # (B, T/2, D)

        x = x[:, -1, :]            # Last compressed timestep

        out = self.output_proj(x)
        out = out.view(-1, self.pred_len, self.input_dim)

        return out


In [84]:
class SeriesDecomposition(nn.Module):
    def __init__(self, kernel_size=25):
        super().__init__()
        self.kernel_size = kernel_size
        self.padding = kernel_size // 2

    def forward(self, x):
        # Moving average (trend)
        trend = nn.functional.avg_pool1d(
            x.transpose(1, 2),
            kernel_size=self.kernel_size,
            stride=1,
            padding=self.padding
        ).transpose(1, 2)

        seasonal = x - trend
        return seasonal, trend


In [109]:
class AutoformerEncoderOnly(nn.Module):
    def __init__(self,
                 input_dim,
                 pred_len,
                 kernel_size,
                 d_model,
                 n_heads,
                 enc_layers,
                 dropout):

        super().__init__()

        self.pred_len = pred_len
        self.input_dim = input_dim

        self.decomp = SeriesDecomposition(kernel_size)

        self.input_proj = nn.Linear(input_dim, d_model)
        self.pos_enc = PositionalEncoding(d_model)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dropout=dropout,
            batch_first=True
        )

        self.encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=enc_layers
        )

        self.output_proj = nn.Linear(d_model, pred_len * input_dim)

    def forward(self, x):

        seasonal, trend = self.decomp(x)

        x = self.input_proj(seasonal)
        x = self.pos_enc(x)
        x = self.encoder(x)

        x = x[:, -1, :]

        out = self.output_proj(x)
        out = out.view(-1, self.pred_len, self.input_dim)

        return out


In [86]:
def train_model(model, train_loader, val_loader, residual=False):

    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
    criterion = nn.MSELoss()

    best_val = float("inf")
    patience=5
    patience_counter=0


    for epoch in range(20):
        model.train()
        train_loss = 0

        for X_batch, Y_batch in train_loader:
            X_batch = X_batch.to(device)
            Y_batch = Y_batch.to(device)

            optimizer.zero_grad()
            output = model(X_batch)

            loss = criterion(output, Y_batch)
            loss.backward()

            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            train_loss += loss.item()

        # Validation
        model.eval()
        val_smape = 0

        with torch.no_grad():
            for X_batch, Y_batch_res in val_loader:

                X_batch = X_batch.to(device)
                Y_batch_res = Y_batch_res.to(device)

                residual_pred = model(X_batch)

                last_value = X_batch[:, -1:, :]
                final_pred = last_value + residual_pred
                Y_original = last_value + Y_batch_res

                val_smape += smape(Y_original, final_pred).item()

        val_smape /= len(val_loader)

        print(f"Epoch {epoch+1} | Train Loss: {train_loss/len(train_loader):.4f} | Val sMAPE: {val_smape:.4f}")

        if val_smape<best_val:
            best_val=val_smape
            patience_counter=0
            torch.save(model.state_dict(),"best_model.pth")
        else:
            patience_counter+=1

        if patience_counter>=patience:
            print("Early stopping triggered")
            break
    model.load_state_dict(torch.load("best_model.pth"))
    return best_val








  


In [ ]:
def build_model(mod_type,input_dim,pred_len,d_model,n_heads,enc_layers,dropout,kernel_size=None):
    if mod_type=="informer":
        model=InformerEncoderOnly(
            input_dim=input_dim,
            pred_len=pred_len,
            d_model=d_model,
            n_heads=n_heads,
            enc_layers=enc_layers,
            dropout=dropout
        )
    elif mod_type=="autoformer":
        model=AutoformerEncoderOnly(
            input_dim=input_dim,
            pred_len=pred_len,
            d_model=d_model,
            n_heads=n_heads,
            enc_layers=enc_layers,
            dropout=dropout,
            kernel_size=kernel_size,
        )
    else:
        raise ValueError("Unknown Model Type")

    return model.to(device)

In [123]:
# Final Caller:seq len-48, pred len-199

seq_len,pred_len,input_dim,d_model,n_heads,enc_layers,dropout=config_setup(seq_len=48,pred_len=199,input_dims=train_data.shape[1],d_model=64,n_heads=4,enc_layers=2,dropout=0.1)

X_train,Y_train_res,X_val,Y_val_res,Y_val=windowcreation_caller(seq_len,pred_len)

train_loader,val_loader=dataloader_call(X_train,Y_train_res, X_val,Y_val_res)
print("---------------Informer Epochs Now:----------------")
model=build_model(mod_type="informer",
                  input_dim=input_dim,
                  pred_len=pred_len,
                  d_model=d_model,
                  n_heads=n_heads,
                  enc_layers=enc_layers,
                  dropout=dropout
)
val_smape_informer_s_48=train_model(model,train_loader,val_loader,residual=True)
print("---------------Autoformer Epochs Now:----------------")
model=build_model(mod_type="autoformer",
                  input_dim=input_dim,
                  pred_len=pred_len,
                  d_model=d_model,
                  n_heads=n_heads,
                  enc_layers=enc_layers,
                  dropout=dropout,
                  kernel_size=23
)
val_smape_autoformer_s_48=train_model(model,train_loader,val_loader,residual=True)

naive_s_48=compute_naive(X_val,Y_val)
print("Naive 199-step sMAPE for 48-input_window:", naive_s_48)
        

---------------Informer Epochs Now:----------------
Epoch 1 | Train Loss: 1345.0202 | Val sMAPE: 0.2132
Epoch 2 | Train Loss: 783.8659 | Val sMAPE: 0.2516
Epoch 3 | Train Loss: 194.3044 | Val sMAPE: 0.3411
Epoch 4 | Train Loss: 2.8683 | Val sMAPE: 0.2724
Epoch 5 | Train Loss: 0.3461 | Val sMAPE: 0.1905
Epoch 6 | Train Loss: 0.3226 | Val sMAPE: 0.1836
Epoch 7 | Train Loss: 0.3144 | Val sMAPE: 0.1814
Epoch 8 | Train Loss: 0.3110 | Val sMAPE: 0.1803
Epoch 9 | Train Loss: 0.3090 | Val sMAPE: 0.1797
Epoch 10 | Train Loss: 0.3076 | Val sMAPE: 0.1789
Epoch 11 | Train Loss: 0.3067 | Val sMAPE: 0.1786
Epoch 12 | Train Loss: 0.3059 | Val sMAPE: 0.1781
Epoch 13 | Train Loss: 0.3052 | Val sMAPE: 0.1780
Epoch 14 | Train Loss: 0.3047 | Val sMAPE: 0.1776
Epoch 15 | Train Loss: 0.3040 | Val sMAPE: 0.1772
Epoch 16 | Train Loss: 0.3033 | Val sMAPE: 0.1770
Epoch 17 | Train Loss: 0.3022 | Val sMAPE: 0.1768
Epoch 18 | Train Loss: 0.3017 | Val sMAPE: 0.1764
Epoch 19 | Train Loss: 0.3010 | Val sMAPE: 0.1761


In [124]:
results["Informer_s_48_p_199"]={
    "model_smape":val_smape_informer_s_48,
    "naive_smape":naive_s_48
}
results["Autoformer_s_48_p_199"]={
    "model_smape":val_smape_autoformer_s_48,
    "naive_smape":naive_s_48
}

In [125]:
comparison=pd.DataFrame(results).T
comparison["beats_naive"]=comparison["model_smape"]<comparison["naive_smape"]
comparison.round(4)

,model_smape,naive_smape,beats_naive
Informer 24,0.0725,0.0753,True
Autoformer 24,0.0728,0.0753,True
Informer 96,0.1236,0.1378,True
Autoformer 96,0.1224,0.1378,True
Informer 199,0.1763,0.1933,True
Autoformer 199,0.1608,0.1933,True
Informer 366,0.2857,0.2546,False
Autoformer 366,0.1988,0.2546,True
Informer_s_48_p_199,0.1760,0.2033,True
Autoformer_s_48_p_199,0.1651,0.2033,True


In [ ]:
# Final Caller:seq len-96, pred len-199

seq_len,pred_len,input_dim,d_model,n_heads,enc_layers,dropout=config_setup(seq_len=96,pred_len=199,input_dims=train_data.shape[1],d_model=64,n_heads=4,enc_layers=2,dropout=0.1)

X_train,Y_train_res,X_val,Y_val_res,Y_val=windowcreation_caller(seq_len,pred_len)

train_loader,val_loader=dataloader_call(X_train,Y_train_res, X_val,Y_val_res)
print("---------------Informer Epochs Now:----------------")
model=build_model(mod_type="informer",
                  input_dim=input_dim,
                  pred_len=pred_len,
                  d_model=d_model,
                  n_heads=n_heads,
                  enc_layers=enc_layers,
                  dropout=dropout
)
val_smape_informer_s_96=train_model(model,train_loader,val_loader,residual=True)
print("---------------Autoformer Epochs Now:----------------")
model=build_model(mod_type="autoformer",
                  input_dim=input_dim,
                  pred_len=pred_len,
                  d_model=d_model,
                  n_heads=n_heads,
                  enc_layers=enc_layers,
                  dropout=dropout,
                  kernel_size=23
)
val_smape_autoformer_s_96=train_model(model,train_loader,val_loader,residual=True)

naive_s_96=compute_naive(X_val,Y_val)
print("Naive 199-step sMAPE for 96-input_window:", naive_s_96)
        

---------------Informer Epochs Now:----------------
Epoch 1 | Train Loss: 1349.3020 | Val sMAPE: 0.2051
Epoch 2 | Train Loss: 795.2884 | Val sMAPE: 0.2536
Epoch 3 | Train Loss: 202.8092 | Val sMAPE: 0.3457
Epoch 4 | Train Loss: 3.4081 | Val sMAPE: 0.2776
Epoch 5 | Train Loss: 0.3464 | Val sMAPE: 0.1962
Epoch 6 | Train Loss: 0.3210 | Val sMAPE: 0.1865
Epoch 7 | Train Loss: 0.3134 | Val sMAPE: 0.1834
Epoch 8 | Train Loss: 0.3099 | Val sMAPE: 0.1816
Epoch 9 | Train Loss: 0.3079 | Val sMAPE: 0.1807
Epoch 10 | Train Loss: 0.3063 | Val sMAPE: 0.1800
Epoch 11 | Train Loss: 0.3055 | Val sMAPE: 0.1796
Epoch 12 | Train Loss: 0.3044 | Val sMAPE: 0.1791
Epoch 13 | Train Loss: 0.3036 | Val sMAPE: 0.1788
Epoch 14 | Train Loss: 0.3029 | Val sMAPE: 0.1783
Epoch 15 | Train Loss: 0.3021 | Val sMAPE: 0.1779
Epoch 16 | Train Loss: 0.3011 | Val sMAPE: 0.1777
Epoch 17 | Train Loss: 0.3006 | Val sMAPE: 0.1770
Epoch 18 | Train Loss: 0.2994 | Val sMAPE: 0.1769
Epoch 19 | Train Loss: 0.2985 | Val sMAPE: 0.1765


In [127]:
results["Informer_s_96_p_199"]={
    "model_smape":val_smape_informer_s_96,
    "naive_smape":naive_s_96
}
results["Autoformer_s_96_p_199"]={
    "model_smape":val_smape_autoformer_s_96,
    "naive_smape":naive_s_96
}

In [128]:
comparison=pd.DataFrame(results).T
comparison["beats_naive"]=comparison["model_smape"]<comparison["naive_smape"]
comparison.round(4)

,model_smape,naive_smape,beats_naive
Informer 24,0.0725,0.0753,True
Autoformer 24,0.0728,0.0753,True
Informer 96,0.1236,0.1378,True
Autoformer 96,0.1224,0.1378,True
Informer 199,0.1763,0.1933,True
Autoformer 199,0.1608,0.1933,True
Informer 366,0.2857,0.2546,False
Autoformer 366,0.1988,0.2546,True
Informer_s_48_p_199,0.1760,0.2033,True
Autoformer_s_48_p_199,0.1651,0.2033,True


In [130]:
# Final Caller:seq len-192, pred len-199

seq_len,pred_len,input_dim,d_model,n_heads,enc_layers,dropout=config_setup(seq_len=192,pred_len=199,input_dims=train_data.shape[1],d_model=64,n_heads=4,enc_layers=2,dropout=0.1)

X_train,Y_train_res,X_val,Y_val_res,Y_val=windowcreation_caller(seq_len,pred_len)

train_loader,val_loader=dataloader_call(X_train,Y_train_res, X_val,Y_val_res)
print("---------------Informer Epochs Now:----------------")
model=build_model(mod_type="informer",
                  input_dim=input_dim,
                  pred_len=pred_len,
                  d_model=d_model,
                  n_heads=n_heads,
                  enc_layers=enc_layers,
                  dropout=dropout
)
val_smape_informer_s_192=train_model(model,train_loader,val_loader,residual=True)
print("---------------Autoformer Epochs Now:----------------")
model=build_model(mod_type="autoformer",
                  input_dim=input_dim,
                  pred_len=pred_len,
                  d_model=d_model,
                  n_heads=n_heads,
                  enc_layers=enc_layers,
                  dropout=dropout,
                  kernel_size=23
)
val_smape_autoformer_s_192=train_model(model,train_loader,val_loader,residual=True)

naive_s_192=compute_naive(X_val,Y_val)
print("Naive 199-step sMAPE for 192-input_window:", naive_s_192)
        

---------------Informer Epochs Now:----------------
Epoch 1 | Train Loss: 1359.5408 | Val sMAPE: 0.1997
Epoch 2 | Train Loss: 825.4339 | Val sMAPE: 0.2509
Epoch 3 | Train Loss: 230.7944 | Val sMAPE: 0.3688
Epoch 4 | Train Loss: 6.0151 | Val sMAPE: 0.3012
Epoch 5 | Train Loss: 0.3494 | Val sMAPE: 0.2063
Epoch 6 | Train Loss: 0.3238 | Val sMAPE: 0.1911
Epoch 7 | Train Loss: 0.3169 | Val sMAPE: 0.1864
Epoch 8 | Train Loss: 0.3133 | Val sMAPE: 0.1846
Epoch 9 | Train Loss: 0.3108 | Val sMAPE: 0.1835
Epoch 10 | Train Loss: 0.3092 | Val sMAPE: 0.1826
Epoch 11 | Train Loss: 0.3080 | Val sMAPE: 0.1821
Epoch 12 | Train Loss: 0.3069 | Val sMAPE: 0.1816
Epoch 13 | Train Loss: 0.3061 | Val sMAPE: 0.1811
Epoch 14 | Train Loss: 0.3052 | Val sMAPE: 0.1809
Epoch 15 | Train Loss: 0.3043 | Val sMAPE: 0.1803
Epoch 16 | Train Loss: 0.3036 | Val sMAPE: 0.1798
Epoch 17 | Train Loss: 0.3026 | Val sMAPE: 0.1796
Epoch 18 | Train Loss: 0.3017 | Val sMAPE: 0.1793
Epoch 19 | Train Loss: 0.3012 | Val sMAPE: 0.1785


In [131]:
results["Informer_s_192_p_199"]={
    "model_smape":val_smape_informer_s_192,
    "naive_smape":naive_s_192
}
results["Autoformer_s_192_p_199"]={
    "model_smape":val_smape_autoformer_s_192,
    "naive_smape":naive_s_192
}

In [ ]:
comparison=pd.DataFrame(results).T
comparison["beats_naive"]=comparison["model_smape"]<comparison["naive_smape"]
comparison.round(4)

,model_smape,naive_smape,beats_naive
Informer 24,0.0725,0.0753,True
Autoformer 24,0.0728,0.0753,True
Informer 96,0.1236,0.1378,True
Autoformer 96,0.1224,0.1378,True
Informer 199,0.1763,0.1933,True
Autoformer 199,0.1608,0.1933,True
Informer 366,0.2857,0.2546,False
Autoformer 366,0.1988,0.2546,True
Informer_s_48_p_199,0.1760,0.2033,True
Autoformer_s_48_p_199,0.1651,0.2033,True


In [136]:
# Final Caller:seq len-256, pred len-199

seq_len,pred_len,input_dim,d_model,n_heads,enc_layers,dropout=config_setup(seq_len=256,pred_len=199,input_dims=train_data.shape[1],d_model=64,n_heads=4,enc_layers=2,dropout=0.1)

X_train,Y_train_res,X_val,Y_val_res,Y_val=windowcreation_caller(seq_len,pred_len)

train_loader,val_loader=dataloader_call(X_train,Y_train_res, X_val,Y_val_res)
print("---------------Informer Epochs Now:----------------")
model=build_model(mod_type="informer",
                  input_dim=input_dim,
                  pred_len=pred_len,
                  d_model=d_model,
                  n_heads=n_heads,
                  enc_layers=enc_layers,
                  dropout=dropout
)
val_smape_informer_s_256=train_model(model,train_loader,val_loader,residual=True)
print("---------------Autoformer Epochs Now:----------------")
model=build_model(mod_type="autoformer",
                  input_dim=input_dim,
                  pred_len=pred_len,
                  d_model=d_model,
                  n_heads=n_heads,
                  enc_layers=enc_layers,
                  dropout=dropout,
                  kernel_size=23
)
val_smape_autoformer_s_256=train_model(model,train_loader,val_loader,residual=True)

naive_s_256=compute_naive(X_val,Y_val)
print("Naive 199-step sMAPE for 256-input_window:", naive_s_256)
        

---------------Informer Epochs Now:----------------
Epoch 1 | Train Loss: 1353.3679 | Val sMAPE: 0.1902
Epoch 2 | Train Loss: 816.0348 | Val sMAPE: 0.2569
Epoch 3 | Train Loss: 226.3702 | Val sMAPE: 0.3797
Epoch 4 | Train Loss: 5.8649 | Val sMAPE: 0.3118
Epoch 5 | Train Loss: 0.3363 | Val sMAPE: 0.2119
Epoch 6 | Train Loss: 0.3175 | Val sMAPE: 0.1940
Early stopping triggered
---------------Autoformer Epochs Now:----------------
Epoch 1 | Train Loss: 1448.1411 | Val sMAPE: 0.1761
Epoch 2 | Train Loss: 1398.0247 | Val sMAPE: 0.1704
Epoch 3 | Train Loss: 1348.0343 | Val sMAPE: 0.1691
Epoch 4 | Train Loss: 1297.4157 | Val sMAPE: 0.1683
Epoch 5 | Train Loss: 1245.5037 | Val sMAPE: 0.1674
Epoch 6 | Train Loss: 1191.9584 | Val sMAPE: 0.1664
Epoch 7 | Train Loss: 1136.6140 | Val sMAPE: 0.1654
Epoch 8 | Train Loss: 1079.5188 | Val sMAPE: 0.1644
Epoch 9 | Train Loss: 1020.7606 | Val sMAPE: 0.1634
Epoch 10 | Train Loss: 960.5158 | Val sMAPE: 0.1623
Epoch 11 | Train Loss: 899.0180 | Val sMAPE: 0.1

In [137]:
results["Informer_s_256_p_199"]={
    "model_smape":val_smape_informer_s_256,
    "naive_smape":naive_s_256
}
results["Autoformer_s_256_p_199"]={
    "model_smape":val_smape_autoformer_s_256,
    "naive_smape":naive_s_256
}

In [138]:
comparison=pd.DataFrame(results).T
comparison["beats_naive"]=comparison["model_smape"]<comparison["naive_smape"]
comparison.round(4)

,model_smape,naive_smape,beats_naive
Informer 24,0.0725,0.0753,True
Autoformer 24,0.0728,0.0753,True
Informer 96,0.1236,0.1378,True
Autoformer 96,0.1224,0.1378,True
Informer 199,0.1763,0.1933,True
Autoformer 199,0.1608,0.1933,True
Informer 366,0.2857,0.2546,False
Autoformer 366,0.1988,0.2546,True
Informer_s_48_p_199,0.1760,0.2033,True
Autoformer_s_48_p_199,0.1651,0.2033,True


In [142]:
# Differentiating Attention heads


# Final Caller:seq len-192, pred len-199

seq_len,pred_len,input_dim,d_model,n_heads,enc_layers,dropout=config_setup(seq_len=192,pred_len=199,input_dims=train_data.shape[1],d_model=60,n_heads=3,enc_layers=2,dropout=0.1)

X_train,Y_train_res,X_val,Y_val_res,Y_val=windowcreation_caller(seq_len,pred_len)

train_loader,val_loader=dataloader_call(X_train,Y_train_res, X_val,Y_val_res)
print("---------------Informer Epochs Now:----------------")
model=build_model(mod_type="informer",
                  input_dim=input_dim,
                  pred_len=pred_len,
                  d_model=d_model,
                  n_heads=n_heads,
                  enc_layers=enc_layers,
                  dropout=dropout
)
val_smape_informer_s_192_N_3=train_model(model,train_loader,val_loader,residual=True)
print("---------------Autoformer Epochs Now:----------------")
model=build_model(mod_type="autoformer",
                  input_dim=input_dim,
                  pred_len=pred_len,
                  d_model=d_model,
                  n_heads=n_heads,
                  enc_layers=enc_layers,
                  dropout=dropout,
                  kernel_size=23
)
val_smape_autoformer_s_192_N_3=train_model(model,train_loader,val_loader,residual=True)

naive_s_192_N_3=compute_naive(X_val,Y_val)
print("Naive 199-step sMAPE for 192-input_window and 3 Attention Heads:", naive_s_192_N_3)
        

---------------Informer Epochs Now:----------------


/var/folders/nc/1fydqg_n407c_n6_t39zsp2w0000gn/T/ipykernel_22609/1984594084.py:21: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.num_heads is odd
  self.encoder = nn.TransformerEncoder(


Epoch 1 | Train Loss: 1365.6655 | Val sMAPE: 0.1974
Epoch 2 | Train Loss: 870.5764 | Val sMAPE: 0.2216
Epoch 3 | Train Loss: 291.0567 | Val sMAPE: 0.3947
Epoch 4 | Train Loss: 16.6373 | Val sMAPE: 0.3175
Epoch 5 | Train Loss: 0.3577 | Val sMAPE: 0.2075
Epoch 6 | Train Loss: 0.3220 | Val sMAPE: 0.1891
Epoch 7 | Train Loss: 0.3143 | Val sMAPE: 0.1841
Epoch 8 | Train Loss: 0.3103 | Val sMAPE: 0.1820
Epoch 9 | Train Loss: 0.3076 | Val sMAPE: 0.1808
Epoch 10 | Train Loss: 0.3058 | Val sMAPE: 0.1800
Epoch 11 | Train Loss: 0.3044 | Val sMAPE: 0.1797
Epoch 12 | Train Loss: 0.3034 | Val sMAPE: 0.1794
Epoch 13 | Train Loss: 0.3025 | Val sMAPE: 0.1789
Epoch 14 | Train Loss: 0.3016 | Val sMAPE: 0.1786
Epoch 15 | Train Loss: 0.3009 | Val sMAPE: 0.1784
Epoch 16 | Train Loss: 0.3005 | Val sMAPE: 0.1779
Epoch 17 | Train Loss: 0.2995 | Val sMAPE: 0.1776
Epoch 18 | Train Loss: 0.2985 | Val sMAPE: 0.1773
Epoch 19 | Train Loss: 0.2979 | Val sMAPE: 0.1767
Epoch 20 | Train Loss: 0.2973 | Val sMAPE: 0.1765
-

/var/folders/nc/1fydqg_n407c_n6_t39zsp2w0000gn/T/ipykernel_22609/2613008536.py:28: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.num_heads is odd
  self.encoder = nn.TransformerEncoder(


Epoch 1 | Train Loss: 1450.1390 | Val sMAPE: 0.1820
Epoch 2 | Train Loss: 1402.5090 | Val sMAPE: 0.1758
Epoch 3 | Train Loss: 1354.8885 | Val sMAPE: 0.1746
Epoch 4 | Train Loss: 1306.6387 | Val sMAPE: 0.1737
Epoch 5 | Train Loss: 1257.0447 | Val sMAPE: 0.1728
Epoch 6 | Train Loss: 1205.7466 | Val sMAPE: 0.1719
Epoch 7 | Train Loss: 1152.6128 | Val sMAPE: 0.1709
Epoch 8 | Train Loss: 1097.6277 | Val sMAPE: 0.1700
Epoch 9 | Train Loss: 1040.8880 | Val sMAPE: 0.1690
Epoch 10 | Train Loss: 982.5663 | Val sMAPE: 0.1681
Epoch 11 | Train Loss: 922.8505 | Val sMAPE: 0.1671
Epoch 12 | Train Loss: 862.0447 | Val sMAPE: 0.1661
Epoch 13 | Train Loss: 800.4738 | Val sMAPE: 0.1652
Epoch 14 | Train Loss: 738.4607 | Val sMAPE: 0.1643
Epoch 15 | Train Loss: 676.4501 | Val sMAPE: 0.1634
Epoch 16 | Train Loss: 614.8055 | Val sMAPE: 0.1626
Epoch 17 | Train Loss: 553.9826 | Val sMAPE: 0.1618
Epoch 18 | Train Loss: 494.4144 | Val sMAPE: 0.1610
Epoch 19 | Train Loss: 436.5435 | Val sMAPE: 0.1604
Epoch 20 | T

In [143]:
results["Informer_s_192_p_199_N_3"]={
    "model_smape":val_smape_informer_s_192_N_3,
    "naive_smape":naive_s_192_N_3
}
results["Autoformer_s_192_p_199_N_3"]={
    "model_smape":val_smape_autoformer_s_192_N_3,
    "naive_smape":naive_s_192_N_3
}

In [ ]:
comparison=pd.DataFrame(results).T
comparison["beats_naive"]=comparison["model_smape"]<comparison["naive_smape"]
comparison.round(4)

,model_smape,naive_smape,beats_naive
Informer 24,0.0725,0.0753,True
Autoformer 24,0.0728,0.0753,True
Informer 96,0.1236,0.1378,True
Autoformer 96,0.1224,0.1378,True
Informer 199,0.1763,0.1933,True
Autoformer 199,0.1608,0.1933,True
Informer 366,0.2857,0.2546,False
Autoformer 366,0.1988,0.2546,True
Informer_s_48_p_199,0.1760,0.2033,True
Autoformer_s_48_p_199,0.1651,0.2033,True


In [146]:
# Differentiating Attention heads


# Final Caller:seq len-192, pred len-199,N_heads=5,d_model=80

seq_len,pred_len,input_dim,d_model,n_heads,enc_layers,dropout=config_setup(seq_len=192,pred_len=199,input_dims=train_data.shape[1],d_model=80,n_heads=5,enc_layers=2,dropout=0.1)

X_train,Y_train_res,X_val,Y_val_res,Y_val=windowcreation_caller(seq_len,pred_len)

train_loader,val_loader=dataloader_call(X_train,Y_train_res, X_val,Y_val_res)
print("---------------Informer Epochs Now:----------------")
model=build_model(mod_type="informer",
                  input_dim=input_dim,
                  pred_len=pred_len,
                  d_model=d_model,
                  n_heads=n_heads,
                  enc_layers=enc_layers,
                  dropout=dropout
)
val_smape_informer_s_192_N_5=train_model(model,train_loader,val_loader,residual=True)
print("---------------Autoformer Epochs Now:----------------")
model=build_model(mod_type="autoformer",
                  input_dim=input_dim,
                  pred_len=pred_len,
                  d_model=d_model,
                  n_heads=n_heads,
                  enc_layers=enc_layers,
                  dropout=dropout,
                  kernel_size=23
)
val_smape_autoformer_s_192_N_5=train_model(model,train_loader,val_loader,residual=True)

naive_s_192_N_5=compute_naive(X_val,Y_val)
print("Naive 199-step sMAPE for 192-input_window and 5 Attention Heads:", naive_s_192_N_5)
        

---------------Informer Epochs Now:----------------


/var/folders/nc/1fydqg_n407c_n6_t39zsp2w0000gn/T/ipykernel_22609/1984594084.py:21: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.num_heads is odd
  self.encoder = nn.TransformerEncoder(


Epoch 1 | Train Loss: 1296.7942 | Val sMAPE: 0.2022
Epoch 2 | Train Loss: 578.9804 | Val sMAPE: 0.4005
Epoch 3 | Train Loss: 50.8500 | Val sMAPE: 0.3427
Epoch 4 | Train Loss: 0.3920 | Val sMAPE: 0.2557
Epoch 5 | Train Loss: 0.3443 | Val sMAPE: 0.2187
Epoch 6 | Train Loss: 0.3319 | Val sMAPE: 0.2060
Early stopping triggered
---------------Autoformer Epochs Now:----------------


/var/folders/nc/1fydqg_n407c_n6_t39zsp2w0000gn/T/ipykernel_22609/2613008536.py:28: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.num_heads is odd
  self.encoder = nn.TransformerEncoder(


Epoch 1 | Train Loss: 1441.5032 | Val sMAPE: 0.1779
Epoch 2 | Train Loss: 1379.5372 | Val sMAPE: 0.1743
Epoch 3 | Train Loss: 1317.4116 | Val sMAPE: 0.1731
Epoch 4 | Train Loss: 1254.6402 | Val sMAPE: 0.1720
Epoch 5 | Train Loss: 1190.5669 | Val sMAPE: 0.1710
Epoch 6 | Train Loss: 1124.8645 | Val sMAPE: 0.1700
Epoch 7 | Train Loss: 1057.4231 | Val sMAPE: 0.1690
Epoch 8 | Train Loss: 988.3859 | Val sMAPE: 0.1680
Epoch 9 | Train Loss: 917.9932 | Val sMAPE: 0.1670
Epoch 10 | Train Loss: 846.5740 | Val sMAPE: 0.1660
Epoch 11 | Train Loss: 774.5555 | Val sMAPE: 0.1651
Epoch 12 | Train Loss: 702.4326 | Val sMAPE: 0.1644
Epoch 13 | Train Loss: 630.7435 | Val sMAPE: 0.1637
Epoch 14 | Train Loss: 560.0912 | Val sMAPE: 0.1631
Epoch 15 | Train Loss: 491.1199 | Val sMAPE: 0.1627
Epoch 16 | Train Loss: 424.4492 | Val sMAPE: 0.1625
Epoch 17 | Train Loss: 360.7675 | Val sMAPE: 0.1624
Epoch 18 | Train Loss: 300.6904 | Val sMAPE: 0.1624
Epoch 19 | Train Loss: 244.8702 | Val sMAPE: 0.1626
Epoch 20 | Tra

In [147]:
results["Informer_s_192_p_199_N_5"]={
    "model_smape":val_smape_informer_s_192_N_5,
    "naive_smape":naive_s_192_N_5
}
results["Autoformer_s_192_p_199_N_5"]={
    "model_smape":val_smape_autoformer_s_192_N_5,
    "naive_smape":naive_s_192_N_5
}

In [148]:
comparison=pd.DataFrame(results).T
comparison["beats_naive"]=comparison["model_smape"]<comparison["naive_smape"]
comparison.round(4)

,model_smape,naive_smape,beats_naive
Informer 24,0.0725,0.0753,True
Autoformer 24,0.0728,0.0753,True
Informer 96,0.1236,0.1378,True
Autoformer 96,0.1224,0.1378,True
Informer 199,0.1763,0.1933,True
Autoformer 199,0.1608,0.1933,True
Informer 366,0.2857,0.2546,False
Autoformer 366,0.1988,0.2546,True
Informer_s_48_p_199,0.1760,0.2033,True
Autoformer_s_48_p_199,0.1651,0.2033,True


In [150]:
# Differentiating kernel size


# Final Caller:seq len-192, pred len-199,N_heads=3,d_model=60

seq_len,pred_len,input_dim,d_model,n_heads,enc_layers,dropout=config_setup(seq_len=192,pred_len=199,input_dims=train_data.shape[1],d_model=60,n_heads=3,enc_layers=2,dropout=0.1)

X_train,Y_train_res,X_val,Y_val_res,Y_val=windowcreation_caller(seq_len,pred_len)

print("---------------Autoformer Epochs Now:----------------")
model=build_model(mod_type="autoformer",
                  input_dim=input_dim,
                  pred_len=pred_len,
                  d_model=d_model,
                  n_heads=n_heads,
                  enc_layers=enc_layers,
                  dropout=dropout,
                  kernel_size=11
)
val_smape_autoformer_s_192_N_3_K_11=train_model(model,train_loader,val_loader,residual=True)

naive_s_192_N_3_K_11=compute_naive(X_val,Y_val)
print("Naive 199-step sMAPE for 192-input_window and 3 Attention Heads:", naive_s_192_N_3_K_11)
        

---------------Autoformer Epochs Now:----------------


/var/folders/nc/1fydqg_n407c_n6_t39zsp2w0000gn/T/ipykernel_22609/2613008536.py:28: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.num_heads is odd
  self.encoder = nn.TransformerEncoder(


Epoch 1 | Train Loss: 1450.4137 | Val sMAPE: 0.1817
Epoch 2 | Train Loss: 1403.1541 | Val sMAPE: 0.1757
Epoch 3 | Train Loss: 1355.6477 | Val sMAPE: 0.1747
Epoch 4 | Train Loss: 1307.4318 | Val sMAPE: 0.1737
Epoch 5 | Train Loss: 1257.8080 | Val sMAPE: 0.1728
Epoch 6 | Train Loss: 1206.5127 | Val sMAPE: 0.1718
Epoch 7 | Train Loss: 1153.3678 | Val sMAPE: 0.1710
Epoch 8 | Train Loss: 1098.3853 | Val sMAPE: 0.1699
Epoch 9 | Train Loss: 1041.6357 | Val sMAPE: 0.1689
Epoch 10 | Train Loss: 983.3158 | Val sMAPE: 0.1680
Epoch 11 | Train Loss: 923.6354 | Val sMAPE: 0.1670
Epoch 12 | Train Loss: 862.8383 | Val sMAPE: 0.1661
Epoch 13 | Train Loss: 801.2673 | Val sMAPE: 0.1652
Epoch 14 | Train Loss: 739.2638 | Val sMAPE: 0.1643
Epoch 15 | Train Loss: 677.2407 | Val sMAPE: 0.1634
Epoch 16 | Train Loss: 615.5748 | Val sMAPE: 0.1625
Epoch 17 | Train Loss: 554.7351 | Val sMAPE: 0.1618
Epoch 18 | Train Loss: 495.1428 | Val sMAPE: 0.1610
Epoch 19 | Train Loss: 437.2482 | Val sMAPE: 0.1604
Epoch 20 | T

In [151]:
# Differentiating kernel size


# Final Caller:seq len-192, pred len-199,N_heads=3,d_model=60

seq_len,pred_len,input_dim,d_model,n_heads,enc_layers,dropout=config_setup(seq_len=192,pred_len=199,input_dims=train_data.shape[1],d_model=60,n_heads=3,enc_layers=2,dropout=0.1)

X_train,Y_train_res,X_val,Y_val_res,Y_val=windowcreation_caller(seq_len,pred_len)

print("---------------Autoformer Epochs Now:----------------")
model=build_model(mod_type="autoformer",
                  input_dim=input_dim,
                  pred_len=pred_len,
                  d_model=d_model,
                  n_heads=n_heads,
                  enc_layers=enc_layers,
                  dropout=dropout,
                  kernel_size=47
)
val_smape_autoformer_s_192_N_3_K_47=train_model(model,train_loader,val_loader,residual=True)

naive_s_192_N_3_K_47=compute_naive(X_val,Y_val)
print("Naive 199-step sMAPE for 192-input_window and 3 Attention Heads:", naive_s_192_N_3_K_47)
        

---------------Autoformer Epochs Now:----------------


/var/folders/nc/1fydqg_n407c_n6_t39zsp2w0000gn/T/ipykernel_22609/2613008536.py:28: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.num_heads is odd
  self.encoder = nn.TransformerEncoder(


Epoch 1 | Train Loss: 1449.5382 | Val sMAPE: 0.1848
Epoch 2 | Train Loss: 1403.1272 | Val sMAPE: 0.1762
Epoch 3 | Train Loss: 1355.9571 | Val sMAPE: 0.1747
Epoch 4 | Train Loss: 1307.9526 | Val sMAPE: 0.1738
Epoch 5 | Train Loss: 1258.6511 | Val sMAPE: 0.1728
Epoch 6 | Train Loss: 1207.6867 | Val sMAPE: 0.1720
Epoch 7 | Train Loss: 1154.9333 | Val sMAPE: 0.1710
Epoch 8 | Train Loss: 1100.5196 | Val sMAPE: 0.1701
Epoch 9 | Train Loss: 1044.2207 | Val sMAPE: 0.1691
Epoch 10 | Train Loss: 986.3123 | Val sMAPE: 0.1682
Epoch 11 | Train Loss: 927.0420 | Val sMAPE: 0.1672
Epoch 12 | Train Loss: 866.6808 | Val sMAPE: 0.1662
Epoch 13 | Train Loss: 805.4648 | Val sMAPE: 0.1653
Epoch 14 | Train Loss: 743.8123 | Val sMAPE: 0.1644
Epoch 15 | Train Loss: 682.0771 | Val sMAPE: 0.1635
Epoch 16 | Train Loss: 620.6970 | Val sMAPE: 0.1627
Epoch 17 | Train Loss: 560.0616 | Val sMAPE: 0.1619
Epoch 18 | Train Loss: 500.6338 | Val sMAPE: 0.1611
Epoch 19 | Train Loss: 442.8363 | Val sMAPE: 0.1605
Epoch 20 | T

In [153]:
results["Autoformer_s_192_p_199_N_3_K_11"]={
    "model_smape":val_smape_autoformer_s_192_N_3_K_11,
    "naive_smape":naive_s_192_N_3_K_11
}
results["Autoformer_s_192_p_199_N_3_K_47"]={
    "model_smape":val_smape_autoformer_s_192_N_3_K_47,
    "naive_smape":naive_s_192_N_3_K_47
}

In [154]:
comparison=pd.DataFrame(results).T
comparison["beats_naive"]=comparison["model_smape"]<comparison["naive_smape"]
comparison.round(4)

,model_smape,naive_smape,beats_naive
Informer 24,0.0725,0.0753,True
Autoformer 24,0.0728,0.0753,True
Informer 96,0.1236,0.1378,True
Autoformer 96,0.1224,0.1378,True
Informer 199,0.1763,0.1933,True
Autoformer 199,0.1608,0.1933,True
Informer 366,0.2857,0.2546,False
Autoformer 366,0.1988,0.2546,True
Informer_s_48_p_199,0.1760,0.2033,True
Autoformer_s_48_p_199,0.1651,0.2033,True


### Model for Univariate currenncy

In [159]:
train_uni=train_data[:, -1:] 
val_uni=val_data[:,-1:]

In [160]:
X_train_u, Y_train_u = create_windows(train_uni, SEQ_LEN, PRED_LEN)
X_val_u,   Y_val_u   = create_windows(val_uni, SEQ_LEN, PRED_LEN)


In [161]:
Y_train_res_u = Y_train_u - X_train_u[:, -1:, :]
Y_val_res_u   = Y_val_u - X_val_u[:, -1:, :]


In [162]:
train_loader_u, val_loader_u = dataloader_call(
    X_train_u, Y_train_res_u,
    X_val_u,   Y_val_res_u
)


In [163]:
model = build_model(
    mod_type="autoformer",
    input_dim=1,
    pred_len=PRED_LEN,
    d_model=d_model,
    n_heads=n_heads,
    enc_layers=enc_layers,
    dropout=dropout,
    kernel_size=23
)


/var/folders/nc/1fydqg_n407c_n6_t39zsp2w0000gn/T/ipykernel_22609/2613008536.py:28: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.num_heads is odd
  self.encoder = nn.TransformerEncoder(


In [164]:
val_smape_uni = train_model(
    model,
    train_loader_u,
    val_loader_u,
    residual=True
)


Epoch 1 | Train Loss: 0.5901 | Val sMAPE: 0.3128
Epoch 2 | Train Loss: 0.5204 | Val sMAPE: 0.4184
Epoch 3 | Train Loss: 0.5696 | Val sMAPE: 0.4439
Epoch 4 | Train Loss: 0.5335 | Val sMAPE: 0.4744
Epoch 5 | Train Loss: 0.5260 | Val sMAPE: 0.4833
Epoch 6 | Train Loss: 0.5135 | Val sMAPE: 0.5050
Early stopping triggered


In [165]:
naive_uni=compute_naive(X_val,Y_val)

In [166]:
results["Autoformer_Univariate_OT"]={
    "model_smape":val_smape_uni,
    "naive_smape":naive_uni
}
 

In [167]:
comparison=pd.DataFrame(results).T
comparison["beats_naive"]=comparison["model_smape"]<comparison["naive_smape"]
comparison.round(4)

,model_smape,naive_smape,beats_naive
Informer 24,0.0725,0.0753,True
Autoformer 24,0.0728,0.0753,True
Informer 96,0.1236,0.1378,True
Autoformer 96,0.1224,0.1378,True
Informer 199,0.1763,0.1933,True
Autoformer 199,0.1608,0.1933,True
Informer 366,0.2857,0.2546,False
Autoformer 366,0.1988,0.2546,True
Informer_s_48_p_199,0.1760,0.2033,True
Autoformer_s_48_p_199,0.1651,0.2033,True


In [169]:
# Alternate dropout


# Final Caller:seq len-192, pred len-199, kernel-23, d_model=64, dropout=0.2

seq_len,pred_len,input_dim,d_model,n_heads,enc_layers,dropout=config_setup(seq_len=192,pred_len=199,input_dims=train_data.shape[1],d_model=64,n_heads=4,enc_layers=2,dropout=0.2)

X_train,Y_train_res,X_val,Y_val_res,Y_val=windowcreation_caller(seq_len,pred_len)

train_loader,val_loader=dataloader_call(X_train,Y_train_res, X_val,Y_val_res)
print("---------------Informer Epochs Now:----------------")
model=build_model(mod_type="informer",
                  input_dim=input_dim,
                  pred_len=pred_len,
                  d_model=d_model,
                  n_heads=n_heads,
                  enc_layers=enc_layers,
                  dropout=dropout
)
val_smape_informer_s_192_d_02=train_model(model,train_loader,val_loader,residual=True)
print("---------------Autoformer Epochs Now:----------------")
model=build_model(mod_type="autoformer",
                  input_dim=input_dim,
                  pred_len=pred_len,
                  d_model=d_model,
                  n_heads=n_heads,
                  enc_layers=enc_layers,
                  dropout=dropout,
                  kernel_size=23
)
val_smape_autoformer_s_192_d_02=train_model(model,train_loader,val_loader,residual=True)

naive_s_192_d_02=compute_naive(X_val,Y_val)
print("Naive 199-step sMAPE for 192-input_window and dropout 0.2:", naive_s_192_d_02)
        

---------------Informer Epochs Now:----------------
Epoch 1 | Train Loss: 1358.7926 | Val sMAPE: 0.2000
Epoch 2 | Train Loss: 819.9698 | Val sMAPE: 0.2497
Epoch 3 | Train Loss: 229.5825 | Val sMAPE: 0.3719
Epoch 4 | Train Loss: 6.6030 | Val sMAPE: 0.3023
Epoch 5 | Train Loss: 0.3526 | Val sMAPE: 0.2057
Epoch 6 | Train Loss: 0.3260 | Val sMAPE: 0.1906
Epoch 7 | Train Loss: 0.3193 | Val sMAPE: 0.1863
Epoch 8 | Train Loss: 0.3153 | Val sMAPE: 0.1845
Epoch 9 | Train Loss: 0.3127 | Val sMAPE: 0.1834
Epoch 10 | Train Loss: 0.3111 | Val sMAPE: 0.1826
Epoch 11 | Train Loss: 0.3097 | Val sMAPE: 0.1820
Epoch 12 | Train Loss: 0.3084 | Val sMAPE: 0.1813
Epoch 13 | Train Loss: 0.3074 | Val sMAPE: 0.1809
Epoch 14 | Train Loss: 0.3066 | Val sMAPE: 0.1805
Epoch 15 | Train Loss: 0.3056 | Val sMAPE: 0.1805
Epoch 16 | Train Loss: 0.3045 | Val sMAPE: 0.1802
Epoch 17 | Train Loss: 0.3035 | Val sMAPE: 0.1795
Epoch 18 | Train Loss: 0.3025 | Val sMAPE: 0.1794
Epoch 19 | Train Loss: 0.3016 | Val sMAPE: 0.1789


In [170]:
results["Informer_s_192_p_199_d_02"]={
    "model_smape":val_smape_informer_s_192_d_02,
    "naive_smape":naive_s_192_d_02
}
results["Autoformer_s_192_p_199_d_02"]={
    "model_smape":val_smape_autoformer_s_192_d_02,
    "naive_smape":naive_s_192_d_02
}

In [171]:
comparison=pd.DataFrame(results).T
comparison["beats_naive"]=comparison["model_smape"]<comparison["naive_smape"]
comparison.round(4)

,model_smape,naive_smape,beats_naive
Informer 24,0.0725,0.0753,True
Autoformer 24,0.0728,0.0753,True
Informer 96,0.1236,0.1378,True
Autoformer 96,0.1224,0.1378,True
Informer 199,0.1763,0.1933,True
Autoformer 199,0.1608,0.1933,True
Informer 366,0.2857,0.2546,False
Autoformer 366,0.1988,0.2546,True
Informer_s_48_p_199,0.1760,0.2033,True
Autoformer_s_48_p_199,0.1651,0.2033,True


### Checking for Lag Features

In [185]:
def add_lags(data, lags=[7, 30, 90]):
    df = pd.DataFrame(data)
    for lag in lags:
        df[f"lag_{lag}"] = df.iloc[:, -1].shift(lag)
    df = df.dropna()
    return df.values


In [186]:
train_lag = add_lags(train_data)
val_lag   = add_lags(val_data)


In [187]:
X_train_lag, Y_train_lag = create_windows(train_lag, 96, 199)
X_val_lag,Y_val_lag=create_windows(val_lag,96,199)

In [189]:
input_dim = 9+ 3 #(original_dimension+number_of_lags)


In [190]:
Y_train_res_lag = Y_train_lag - X_train_lag[:, -1:, :]
Y_val_res_lag   = Y_val_lag - X_val_lag[:, -1:, :]


In [191]:
train_loader_lag, val_loader_lag = dataloader_call(
    X_train_lag, Y_train_res_lag,
    X_val_lag,   Y_val_res_lag
)


In [192]:
model = build_model(
    mod_type="autoformer",
    input_dim=input_dim,
    pred_len=199,
    d_model=64,
    n_heads=4,
    enc_layers=2,
    dropout=0.1,
    kernel_size=23
)


In [193]:
val_smape_lag = train_model(
    model,
    train_loader_lag,
    val_loader_lag,
    residual=True
)


Epoch 1 | Train Loss: 1086.4862 | Val sMAPE: 0.1806
Epoch 2 | Train Loss: 1048.8615 | Val sMAPE: 0.1761
Epoch 3 | Train Loss: 1011.1620 | Val sMAPE: 0.1753
Epoch 4 | Train Loss: 972.9299 | Val sMAPE: 0.1748
Epoch 5 | Train Loss: 933.7187 | Val sMAPE: 0.1741
Epoch 6 | Train Loss: 893.2315 | Val sMAPE: 0.1735
Epoch 7 | Train Loss: 851.4245 | Val sMAPE: 0.1728
Epoch 8 | Train Loss: 808.2792 | Val sMAPE: 0.1722
Epoch 9 | Train Loss: 763.8681 | Val sMAPE: 0.1715
Epoch 10 | Train Loss: 718.3727 | Val sMAPE: 0.1709
Epoch 11 | Train Loss: 671.9093 | Val sMAPE: 0.1703
Epoch 12 | Train Loss: 624.7406 | Val sMAPE: 0.1697
Epoch 13 | Train Loss: 577.1194 | Val sMAPE: 0.1692
Epoch 14 | Train Loss: 529.3451 | Val sMAPE: 0.1687
Epoch 15 | Train Loss: 481.7349 | Val sMAPE: 0.1682
Epoch 16 | Train Loss: 434.6146 | Val sMAPE: 0.1678
Epoch 17 | Train Loss: 388.3272 | Val sMAPE: 0.1674
Epoch 18 | Train Loss: 343.2451 | Val sMAPE: 0.1671
Epoch 19 | Train Loss: 299.7153 | Val sMAPE: 0.1669
Epoch 20 | Train L

In [194]:
naive_lag=compute_naive(X_val,Y_val)
print("Naive 199-step sMAPE for 96-input_window and lag features", naive_lag)

Naive 199-step sMAPE for 96-input_window and lag features 0.1885463446378708


In [195]:
results["autoformer_with_lag features"]={
    "model_smape":val_smape_lag,
    "naive_smape":naive_lag
}

In [196]:
comparison=pd.DataFrame(results).T
comparison["beats_naive"]=comparison["model_smape"]<comparison["naive_smape"]
comparison.round(4)

,model_smape,naive_smape,beats_naive
Informer 24,0.0725,0.0753,True
Autoformer 24,0.0728,0.0753,True
Informer 96,0.1236,0.1378,True
Autoformer 96,0.1224,0.1378,True
Informer 199,0.1763,0.1933,True
Autoformer 199,0.1608,0.1933,True
Informer 366,0.2857,0.2546,False
Autoformer 366,0.1988,0.2546,True
Informer_s_48_p_199,0.1760,0.2033,True
Autoformer_s_48_p_199,0.1651,0.2033,True
